<a href="https://colab.research.google.com/github/MortezaHosseini1457/sec-financial-rag/blob/main/sec_financial_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
pip install edgartools unstructured[all_docs] sentence_transformers langchain-openai faiss-cpu langchain langchain_community langchain-huggingface


In [10]:
from unstructured.partition.md import partition_md
from unstructured.chunking.title import chunk_by_title
from edgar import Company, set_identity
from sentence_transformers import SentenceTransformer

set_identity("morteza hosseini smhsmh1998@gmail.com")

filing = Company("NVDA").get_filings(form="10-K").latest()
md = filing.markdown()
elements = partition_md(
        text=md
    )

chunks = chunk_by_title(
        elements,
        max_characters=1500,
        new_after_n_chars=1200,
        overlap=100,
    )

documents = []

for chunk in chunks:
    documents.append({
        "text": chunk.text,
        "metadata": chunk.metadata.to_dict()
    })

print(documents[25])

{'text': 'Our networking offerings include NVLink interconnects and switches, end-to-end platforms for InfiniBand and Ethernet, consisting of network adapters, cables, DPUs, switch chips and systems, as well as software. This has enabled us to architect data center-scale computing platforms that can interconnect up to hundreds of thousands of compute nodes with high-performance networking. Fueled by an expansion in AI and HPC workloads, the data center has become the new unit of computing, with networking as an integral part. In fiscal year 2026, we introduced NVIDIA NVLink Fusion to enable hyperscalers and custom ASIC designers to integrate custom CPUs and XPUs with our platform.\n\nOur customers include all major public and private cloud providers, AI model makers, enterprises and startups, and public sector entities. We work with industry leaders to help build or transform their applications and data center infrastructure. Some of our direct customers include original equipment manu

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

langchain_documents = [
    Document(
        page_content=doc["text"],
        metadata=doc["metadata"]
    )
    for doc in documents
]

vector_store = FAISS.from_documents(
    langchain_documents,
    embeddings
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [23]:
def retrieve_context(query, k=5):
    docs = vector_store.similarity_search(query, k=k)

    context = "\n\n".join(
    f"""
    SOURCE:
    {doc.metadata}

    CONTENT:
    {doc.page_content}
    """
    for doc in docs
    )

    return context

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4-Flash-0731",
    api_key="dahl_DESp7dLNP3hBjdAD82E2jEEK8GYAPAxwg",
    base_url="https://inference.dahl.global/v1",
)

response = llm.invoke("Hello")

print(response.content)

Hello! How can I help you today? If you have any questions or need assistance, feel free to ask. 😊

### 扩展对话提示
如果你希望继续聊，可以告诉我：
- 你想讨论的主题（比如科技、生活、学习等）
- 需要解决的问题
- 或者只是想随意聊聊

我随时在这里！


In [34]:
def ask_rag(question):

    context = retrieve_context(question)

    prompt = f"""
    You are a financial research assistant.

    Answer the question using ONLY the provided context.

    For every factual claim, identify the source chunk that supports it.
    If the answer requires calculation, show the calculation briefly.

    If the information is not available in the context, say:
    "I don't have enough information in the provided documents."

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [33]:
answer = ask_rag(
    "What are the main risks NVIDIA faces according to its 10-K?"
)

print(answer)

Based on the provided documents, the main risks NVIDIA faces are:

- Risks related to business trends influenced by climate change concerns, as the business "could be negatively impacted by concerns around the high absolute energy requirements of our GPUs, despite their much more energy efficient design and operation relative to alternative computing platforms." (Source 2)
- Risks that NVIDIA "may not be able to realize the potential benefits of business investments or acquisitions, and may not be able to successfully integrate acquired companies, which could hurt our ability to grow our business, develop new products or sell our products." (Source 2)
- Risks from government mandates and regulations. For example, "license conditions" imposed by the USG "limit the ability of foreign firms to create and offer as a service large-scale GPU clusters." Additionally, "government mandates in chip designs could introduce system vulnerabilities and expose us to significant risk and potential lia

In [ ]:
pip install edgartools unstructured[all_docs] sentence_transformers langchain-openai faiss-cpu langchain langchain_community langchain-huggingface

from unstructured.partition.md import partition_md
from unstructured.chunking.title import chunk_by_title
from edgar import Company, set_identity
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI


set_identity("morteza hosseini smhsmh1998@gmail.com")

filing = Company("NVDA").get_filings(form="10-K").latest()
md = filing.markdown()
elements = partition_md(
        text=md
    )

chunks = chunk_by_title(
        elements,
        max_characters=1500,
        new_after_n_chars=1200,
        overlap=100,
    )

documents = []

for chunk in chunks:
    documents.append({
        "text": chunk.text,
        "metadata": chunk.metadata.to_dict()
    })


embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

langchain_documents = [
    Document(
        page_content=doc["text"],
        metadata=doc["metadata"]
    )
    for doc in documents
]

vector_store = FAISS.from_documents(
    langchain_documents,
    embeddings
)

def retrieve_context(query, k=5):
    docs = vector_store.similarity_search(query, k=k)

    context = "\n\n".join(
    f"""
    SOURCE:
    {doc.metadata}

    CONTENT:
    {doc.page_content}
    """
    for doc in docs
    )

    return context


llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4-Flash-0731",
    api_key="dahl_DESp7dLNP3hBjdAD82E2jEEK8GYAPAxwg",
    base_url="https://inference.dahl.global/v1",
)

def ask_rag(question):

    context = retrieve_context(question)

    prompt = f"""
    You are a financial research assistant.

    Answer the question using ONLY the provided context.

    For every factual claim, identify the source chunk that supports it.
    If the answer requires calculation, show the calculation briefly.

    If the information is not available in the context, say:
    "I don't have enough information in the provided documents."

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

answer = ask_rag(
    "What are the main risks NVIDIA faces according to its 10-K?"
)

print(answer)